# NN3 12개 지도 평가 — 라운드 정책이 실제 신호가 있는 지도에도 남나

라운드 2 정책(N5 + 학생 2개)을 NN12와 같은 46조각 × seed 5개(230판)에서 돌려 N5와 짝으로 비교합니다. 학습은 없고, 라운드 3 후보 고르기에도 쓰지 않습니다.
라운드 3이 채택되면 라운드 3 정책(230판)도 잽니다. 방법과 읽는 법은 `results/explore/nn3r3_20261105/사전등록.md`에 고정했습니다.

**이 노트북은 두 번 실행합니다.**

1. **지금, 또는 런타임이 비는 대로:** 런타임 유형 **CPU + 높은 RAM(CPU 8개)** 을 고르고 **런타임 → 모두 실행**. 묶음 검증 → 12개 지도 자료 → 시험 → 점검(약 5분) → 라운드 2 정책 230판(약 1~2시간, 어림) → 임시 판정.
2. **`NN3_P_R3_judge`가 끝난 뒤:** 다시 **모두 실행**. 끝난 판은 건너뛰고, 채택이면 라운드 3 정책 230판을 돌린 뒤 최종 판정을 냅니다. 미채택이면 라운드 2 정책만으로 판정합니다.

끊기면 다시 **모두 실행**하세요. 끝나면 "끝났어"라고 알려 주세요. 결과: Drive `FutureTraffic/explore_nn3/map/`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, zipfile, hashlib

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_NN3R3.zip"
NET12_ZIP = DRIVE / "FutureTraffic_network12.zip"
WORKSPACE = Path("/content/FutureTraffic_NN3R3")
N3 = DRIVE / "explore_nn3"
OUT = N3 / "map"
NN12, JOINT2 = DRIVE / "explore_nn12", DRIVE / "explore_joint2"       # 읽기만
R3_JUDGE = N3 / "r3" / "judge" / "judge.json"
MIN_CPUS = 8
for path in (ARCHIVE, NET12_ZIP, N3 / "r2" / "students" / "pool" / "seed_42" / "model.pt", NN12 / "N5" / "runs", NN12 / "TM4" / "runs", JOINT2, OUT / "R2W" / "runs"):
    assert path.exists(), f"필요한 파일·폴더가 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 결과 폴더:", OUT)
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요.")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "NN3R3_V1", "NN3R3 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("nn3r3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON0 설정 {package_manifest['platoon0_config_sha256'][:12]} | 측정 계획 {package_manifest['plan_sha256'][:12]}")

In [ ]:
# 12개 지도 자료(기존 묶음 그대로). 코드는 들어 있지 않고 수요·지도만 있습니다.
with zipfile.ZipFile(NET12_ZIP) as z:
    z.extractall(WORKSPACE)
m12 = json.loads((WORKSPACE / "network12_manifest.json").read_text())
bad = [p for p, h in m12["files"].items() if hashlib.sha256((WORKSPACE / p).read_bytes()).hexdigest() != h]
assert not bad, f"12개 지도 자료 해시 불일치: {bad[:3]}"
assert m12["network12_fingerprint"] == "711875f6c939", "12개 지도 지문이 다릅니다"
print(f"12개 지도 자료: 파일 {len(m12['files'])}개 | 배경 {m12['background']:.0%} | 유턴 {m12['uturn']}")
os.chdir(WORKSPACE)

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_nn3map.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 실행 전 점검(약 5분): 학생을 뺀 이번 코드가 저장된 NN12 N5 판과 소수점까지 같은지, 라운드 2 정책 한 판이 건강한지 봅니다.
CHECK = OUT / "check"
result = CHECK / "check.json"
if not (colab.read_json(result) or {}).get("ok"):
    colab.run_live(["src/explore_nn3map.py", "check", "--n3", str(N3), "--stored-n5", str(NN12 / "N5"), "--out", str(CHECK), "--work", "/content/n3mc"], WORKSPACE, check=False)
r = colab.read_json(result)
assert r and r.get("ok"), f"점검 실패 — 본 실행을 하지 않습니다. 이 칸의 출력을 저에게 보내 주세요: {r and r.get('checks')}"
print("점검 통과:", r["checks"], "| 판 초", r["seconds"])

In [ ]:
# 본 실행(남은 판만). 라운드 3 판정이 채택이면 라운드 3 정책도 돕니다. 백그라운드로 돌아 셀을 멈춰도 계속 돕니다.
judge = colab.read_json(R3_JUDGE) or {}
adopted = judge.get("status") == "done" and judge["reading"]["adopted"]
policies = ["R2W", *(["R3W"] if adopted else [])]
print("이번에 돌릴 정책:", policies, "| 라운드 3 판정:", judge.get("status", "아직 없음"), "| 채택:", adopted)

for policy in policies:
    logs = OUT / "logs" / policy
    def describe(name, folder=OUT / policy / "runs"):
        done = len(list(folder.glob("*/metrics.json")))
        return f"끝난 판 {done}/230, 실패 기록 {len(list(folder.glob('*.failed.json')))}"
    args = ["src/explore_nn3map.py", "run", "--policy", policy, "--n3", str(N3), "--out", str(OUT / policy), "--work", f"/content/n3m_{policy}", "--workers", str(CPUS)]
    if policy == "R3W":
        args += ["--r3-judge", str(R3_JUDGE)]
    pids = colab.launch({f"nn3map_{policy}": args}, WORKSPACE, logs)
    code = colab.watch(pids, logs, 300, describe)
    assert code == 0, "본 실행에서 오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 판은 건너뜁니다)."
    print(policy, "끝.", describe(policy))

In [ ]:
# 판정(SUMO 없음). 작은 파일을 많이 읽어 10~30분쯤 걸릴 수 있습니다.
args = ["src/explore_nn3map.py", "judge", "--root", str(OUT), "--nn12", str(NN12), "--joint2", str(JOINT2), "--out", str(OUT / "judge")]
if R3_JUDGE.exists():
    args += ["--r3-judge", str(R3_JUDGE)]
code = colab.run_live(args, WORKSPACE, check=False)
if (OUT / "judge" / "summary.md").exists():
    display(Markdown((OUT / "judge" / "summary.md").read_text()))
print("판정 코드:", code, "(0이면 끝)")
if not R3_JUDGE.exists():
    print("라운드 3 판정이 아직 없어 임시 판정입니다. NN3_P_R3_judge가 끝난 뒤 이 노트북을 다시 모두 실행하세요.")